# Chiyoda Getting Started

Load `scenarios/station_baseline.yaml`, run one simulation, and inspect evacuation timing plus path-usage telemetry.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

from chiyoda._logging import get_logger
from chiyoda.scenarios.manager import ScenarioManager

get_logger().setLevel(logging.WARNING)

ROOT = Path.cwd()
if not (ROOT / "scenarios").exists():
    ROOT = ROOT.parent

SCENARIO = ROOT / "scenarios" / "station_baseline.yaml"
payload = yaml.safe_load(SCENARIO.read_text())
scenario = payload["scenario"]
simulation = ScenarioManager().build_simulation(scenario)
simulation.run()

{
    "scenario": scenario["name"],
    "agents": len(simulation.agents),
    "steps": simulation.current_step,
    "evacuated": len(simulation.completed_agents),
}

In [ ]:
exit_times_s = np.array(simulation.evacuated_at_step, dtype=float) * simulation.config.dt

fig, ax = plt.subplots(figsize=(7, 4))
if exit_times_s.size:
    x = np.sort(exit_times_s)
    y = np.arange(1, len(x) + 1, dtype=float) / len(simulation.agents)
    ax.step(x, y, where="post", color="#2563eb", linewidth=2)
else:
    ax.text(0.5, 0.5, "No evacuations recorded", ha="center", va="center")
ax.set_xlabel("time (s)")
ax.set_ylabel("evacuated share")
ax.set_title("Exit-time CDF")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.25)
fig.tight_layout()

In [ ]:
heatmap = simulation.path_usage_grid

fig, ax = plt.subplots(figsize=(9, 4.5))
image = ax.imshow(heatmap, origin="lower", cmap="magma")
ax.set_title("Path-usage heatmap")
ax.set_xlabel("x cell")
ax.set_ylabel("y cell")
fig.colorbar(image, ax=ax, label="visits")
fig.tight_layout()

In [ ]:
latest = simulation.step_history[-1]
{
    "time_s": latest.time_s,
    "remaining": latest.remaining,
    "mean_density": latest.mean_density,
    "mean_speed": latest.mean_speed,
    "global_entropy": latest.global_entropy,
}